### Analisis Komparatif: Random Forest, XGBoost, dan K-Nearest Neighbors (KNN)
**Tujuan:** Membandingkan kinerja algoritma *tree-based* dan *distance-based* pada dataset klasifikasi biner.
**Dataset:** Breast Cancer Wisconsin (Diagnostic) - UCI Machine Learning Repository.

#### 1. Import Pustaka
Memuat pustaka standar untuk manipulasi struktur data, komputasi numerik, visualisasi, dan algoritma *Machine Learning*.

In [ ]:
import pandas as pd                                  # Digunakan untuk manipulasi struktur data tabuler (DataFrame)
import numpy as np                                   # Digunakan untuk operasi komputasi numerik dasar dan matriks
import matplotlib.pyplot as plt                      # Pustaka utama untuk merender plot/visualisasi 2D
import seaborn as sns                                # Pustaka visualisasi tingkat tinggi berbasis matplotlib

from sklearn.datasets import load_breast_cancer      # Memuat dataset standar bawaan dari scikit-learn
from sklearn.model_selection import train_test_split # Fungsi pemisah dataset menjadi partisi latih (train) dan uji (test)
from sklearn.preprocessing import StandardScaler       # Modul normalisasi data (Z-score normalization)
from sklearn.ensemble import RandomForestClassifier  # Implementasi algoritma Random Forest
from sklearn.neighbors import KNeighborsClassifier   # Implementasi algoritma K-Nearest Neighbors
from xgboost import XGBClassifier                    # Implementasi algoritma XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # Modul ekstraksi metrik evaluasi model

#### 2. Ekstraksi Data
Mengambil dataset mentah dan mengubahnya menjadi representasi matriks (DataFrame) agar komputasi dan analisis lebih efisien.

In [ ]:
data = load_breast_cancer()                          # Memanggil fungsi pemuat dataset ke dalam memori
X = pd.DataFrame(data.data, columns=data.feature_names) # Mengekstraksi fitur (variabel independen) ke dalam bentuk DataFrame
y = data.target                                      # Mengekstraksi label (variabel target) berupa nilai biner 0/1

print(f"Dimensi Matriks Fitur: {X.shape}")           # Mencetak jumlah baris dan kolom dataset
X.head()                                             # Menampilkan 5 sampel baris pertama untuk inspeksi integritas data

#### 3. Pra-pemrosesan Data (*Preprocessing*)
Pemisahan data mutlak diperlukan untuk menguji model pada data yang belum pernah dilihat (mencegah *data leakage*). Standardisasi wajib diaplikasikan pada algoritma berbasis perhitungan jarak spasial (seperti KNN) agar fitur dengan rentang nilai besar tidak mendominasi fitur lainnya.

In [ ]:
# Membagi data: 80% alokasi latih, 20% alokasi uji. Parameter random_state memastikan reprodusibilitas (hasil konsisten)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()                            # Menginisiasi objek standardisasi (target: mean=0, variance=1)
X_train_scaled = scaler.fit_transform(X_train)       # Menghitung parameter distribusi dari data latih, lalu mentransformasikannya
X_test_scaled = scaler.transform(X_test)             # Mentransformasi data uji menggunakan basis distribusi dari data latih

#### 4. Inisialisasi & Eksekusi Model (*Training*)
Membangun arsitektur model klasifikasi dan melakukan injeksi data latih untuk proses pembelajaran.

In [ ]:
# Deklarasi hyperparameter algoritma
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)      # RF berisikan 100 pohon keputusan independen
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss') # XGBoost dikonfigurasi dengan metrik evaluasi logloss
knn_model = KNeighborsClassifier(n_neighbors=5)                           # KNN akan mengevaluasi proksimitas 5 tetangga terdekat

# Eksekusi optimasi model (fitting)
rf_model.fit(X_train, y_train)                                            # Melatih arsitektur RF menggunakan data asli
xgb_model.fit(X_train, y_train)                                           # Melatih arsitektur XGBoost menggunakan data asli
knn_model.fit(X_train_scaled, y_train)                                    # Melatih arsitektur KNN menggunakan data terskala

#### 5. Prediksi dan Analisis Metrik Kinerja
Memvalidasi kapasitas generalisasi model dengan mengumpankan data uji.

In [ ]:
models = {'Random Forest': rf_model, 'XGBoost': xgb_model, 'KNN': knn_model} # Membungkus objek model dalam dictionary untuk iterasi
predictions = {}                                                             # Inisiasi wadah kosong untuk menyimpan array prediksi

for name, model in models.items():                                           # Iterasi eksekusi dan komputasi per model
    if name == 'KNN':                                                        # Instruksi kondisional khusus algoritma distance-based
        pred = model.predict(X_test_scaled)                                  # Mengumpankan data uji yang telah distandardisasi
    else:
        pred = model.predict(X_test)                                         # Mengumpankan data uji asli untuk tree-based model
        
    predictions[name] = pred                                                 # Menempatkan hasil array klasifikasi ke dalam wadah
    
    # Konstruksi pelaporan performa sistem
    print(f"\n--- Laporan Kinerja: {name} ---")
    print(f"Akurasi Sistem: {accuracy_score(y_test, pred):.4f}")             # Kalkulasi rasio prediksi yang secara absolut benar
    print("Metrik Klasifikasi Rinci:\n", classification_report(y_test, pred)) # Merender kalkulasi presisi, recall, dan f1-score

#### 6. Pemetaan Spatial: Visualisasi Confusion Matrix
Matriks ini digunakan untuk mendiagnosis titik kegagalan spesifik model (membedakan tipe eror False Positives dan False Negatives).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))                               # Mendeklarasikan figur kanvas (1 baris, 3 kolom)

for ax, (name, pred) in zip(axes, predictions.items()):                      # Pemetaan area kanvas dengan setiap keluaran model
    cm = confusion_matrix(y_test, pred)                                      # Komputasi matriks aktual vs prediksi
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)                # Membangun heatmap beranotasi angka diskrit (integer)
    ax.set_title(f'Confusion Matrix: {name}')                                # Membubuhkan label judul pada spesifik plot
    ax.set_xlabel('Prediksi Model')                                          # Mendeklarasikan label korelasi sumbu X
    ax.set_ylabel('Data Aktual')                                             # Mendeklarasikan label korelasi sumbu Y

plt.tight_layout()                                                           # Mengkalibrasi ulang margin agar tidak bertumpuk
plt.show()                                                                   # Mengeksekusi instruksi render grafik ke antarmuka